In [174]:
	
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sys
sys.path.append("../")

from shared_utils.generate import format_conversation, transform_conversations
from early_exit.util import module_name_is_layer_base
import numpy as np

from shared_utils.data import CSVPromptDataset
from shared_utils.load import get_model, get_tokenizer, configs_from_yaml
from shared_utils.generate import generate_text

from early_exit.patching import replace_attention_layers, set_transformer_early_exit_mode

from shared_utils.load import get_model, get_tokenizer, configs_from_yaml
import random

import torch
from torch.optim import Adam
from torch.nn import functional as F
from torch.utils.data import DataLoader

import sys
sys.path.append("../")

from shared_utils.data import CSVPromptDataset
from shared_utils.load import get_model, get_tokenizer, configs_from_yaml
from shared_utils.generate import generate_text

from early_exit.patching import replace_attention_layers, set_transformer_early_exit_mode

import wandb
import pandas as pd
import numpy as np
import string
import html
import matplotlib.colors as mcolors

In [102]:
import torch.nn as nn
from collections import defaultdict
from functools import partial

class ActivationLens:
    """
    A utility class to hook multiple layers of a PyTorch model and collect their
    activations during a forward pass. It is designed for analyses like "Logic Lens,"
    where you want to inspect the intermediate representations of a model.

    The class can be used as a context manager to ensure hooks are automatically removed.

    Attributes:
        activations (defaultdict): A dictionary mapping layer_path (str) to a list
                                   of activation tensors from that layer.
    """

    def __init__(self):
        """Initializes the ActivationLens."""
        self.activations = defaultdict(list)
        self._hook_handles = []
        self._model = None

    def _create_hook_fn(self, layer_path: str):
        """
        Factory function to create a hook function for a specific layer.
        The created hook function knows its layer_path and stores the activation
        in the correct place in our `activations` dictionary.
        """
        def _hook_fn(module, input_tensors, output_tensor):
            # The output of some layers might be a tuple; we're often interested in the first element.
            activation = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
            self.activations[layer_path].append(activation.detach().cpu())
        return _hook_fn

    def register(self, model: nn.Module, layer_paths: list[str]):
        """
        Registers forward hooks to a list of specific layers within the model.

        Args:
            model (nn.Module): The model to hook.
            layer_paths (list[str]): A list of dot-separated string paths to the target layers.
        """
        self._model = model
        self.remove_hooks() # Clear any existing hooks before registering new ones

        for path in layer_paths:
            try:
                # Navigate to the target layer
                target_layer = model
                for part in path.split('.'):
                    target_layer = getattr(target_layer, part)

                # Register the hook and store the handle
                hook_fn = self._create_hook_fn(path)
                handle = target_layer.register_forward_hook(hook_fn)
                self._hook_handles.append(handle)
                print(f"✅ Hook registered on '{type(target_layer).__name__}' at: {path}")

            except AttributeError:
                print(f"⚠️ Error: Could not find layer at path: {path}. Skipping.")
    
    def remove_hooks(self):
        """Removes all registered hooks."""
        for handle in self._hook_handles:
            handle.remove()
        self._hook_handles = []

    def clear_activations(self):
        """Clears all collected activations, but leaves the hooks in place."""
        self.activations.clear()

    # --- Context Manager Methods for clean, automatic hook removal ---
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # When the `with` block is exited, automatically remove all hooks
        self.remove_hooks()
        print("\n✨ All hooks automatically removed.")

In [103]:
# LOAD IN EXPERIMENT ARGS
# num_epoch = 1                     # args.num_epoch
num_exit_samples = 1                  # args.num_exit_samples
device = "cpu"                    # args.device
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"                    # args.model_name
model_config_path = "../config_deepseek.yaml"                     # args.model_config_path
dataset_path = "../results_and_data/early_exit_sft_dataset/test/data.csv"                  # args.dataset_path
prompt_config_path = "../results_and_data/early_exit_sft_dataset/test/prompt_config.json"                    # args.prompt_config_path
batch_size = 1                    # args.batch_size -- might want to sort out batching, but increasing num_exit_samples might be better + less effort

# LOAD IN THE MODEL AND TOKENIZER
tokenizer = get_tokenizer(model_name)
config = configs_from_yaml(model_config_path, tokenizer.eos_token_id)
model = get_model(model_name, config['model'], device)


# LOAD IN DATASET
dataset = CSVPromptDataset(dataset_path, prompt_config_path)
dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=dataset.collate_fn, shuffle=True)


# ENABLE EARLY EXITING
model = replace_attention_layers(model, config['lora'], device)

replacing generate_layer_type_without_early_exit_decision_head layer model.layers.0
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.1
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.2
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.3
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.4
replacing layer model.layers.5
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.6
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.7
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.8
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.9
replacing layer model.layers.10
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.11
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.12
replacing g

In [104]:
set_transformer_early_exit_mode(model, 'sft_teacher')


In [105]:
import torch
import torch.nn as nn
from collections import defaultdict

# --- 1. Minimal Class to Collect Activations ---

class ActivationLens:
    """A minimal class to hook model layers and collect activations."""
    def __init__(self):
        self.activations = defaultdict(list)
        self._hook_handles = []

    def _create_hook_fn(self, layer_path: str):
        """Creates a hook function that saves the output of a specific layer."""
        def _hook_fn(module, input, output):
            # The actual activation tensor is often the first element of the output
            activation = output[0] if isinstance(output, tuple) else output
            self.activations[layer_path].append(activation.detach().cpu())
        return _hook_fn

    def register(self, model: nn.Module, layer_paths: list[str]):
        """Registers a forward hook on each layer in the list."""
        for path in layer_paths:
            try:
                target_layer = model
                for part in path.split('.'):
                    target_layer = getattr(target_layer, part)
                handle = target_layer.register_forward_hook(self._create_hook_fn(path))
                self._hook_handles.append(handle)
            except AttributeError:
                print(f"⚠️ Warning: Could not find layer at path: {path}. Skipping.")
    
    def remove_hooks(self):
        """Removes all registered hooks to clean up."""
        for handle in self._hook_handles:
            handle.remove()
        self._hook_handles = []

# --- 2. Main Script to Generate and Print Outputs ---

# NOTE: Make sure your `model`, `tokenizer`, `config`, `generate_text`, and `device`
# variables are already defined and loaded.

# Define all layers to inspect (0-27 plus the final normalization)
num_layers = 28
layer_paths_to_hook = [f'base_model.model.model.layers.{i}' for i in range(num_layers)]
layer_paths_to_hook.append('base_model.model.model.norm')

# Instantiate the lens and run the model once to collect all activations
lens = ActivationLens()
lens.register(model, layer_paths_to_hook)

prompt = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
system_prompt = "You are a helpful assistant."
prefiller = ""

print("\n--- Running Model to Collect Activations ---")
with torch.no_grad():
    # We only need the model to run; the activations are collected by the hooks
    decoded_response, _ = generate_text(
        model=model,
        prompt=prompt,
        system_prompt=system_prompt,
        prefiller=prefiller,
        tokenizer=tokenizer,
        generation_config=config['generation'],
        device=device
    )
print("--- Model Run Complete ---\n")

# --- 3. Process and Save Output from Selected Layers to DataFrame ---

print("="*40)
print("--- Processing Layers (mod 5 + Final) ---")
print("="*40 + "\n")

# Sort the layers numerically for a clean printout
sorted_layers = sorted(
    lens.activations.keys(),
    key=lambda x: int(x.split('.')[4]) if 'layers' in x else float('inf')
)

# Store results in a list for DataFrame
layer_outputs_data = []

for path in sorted_layers:
    layer_activations = lens.activations[path]
    if not layer_activations:
        continue

    # Get the layer number for filtering
    if 'layers' in path:
        layer_num = int(path.split('.')[4])
        layer_label = f"Layer {layer_num}"
        # Only process layers where layer_num % 5 == 0
        if layer_num % 5 != 0:
            continue
    else:
        # Skip Final Norm layer
        continue

    # Concatenate hidden states from all generation steps into one tensor
    full_sequence_hidden_states = torch.cat(layer_activations, dim=1).to(device)

    # Use the model's readout head to get token probabilities (logits)
    logits = model.early_exit_hidden_state_readout(full_sequence_hidden_states)
    
    # Find the most likely token ID for each position in the sequence
    predicted_token_ids = logits.argmax(-1)
    
    # Decode the sequence of token IDs into human-readable text
    text_from_layer = tokenizer.decode(predicted_token_ids[0], skip_special_tokens=True)

    # Store in list WITHOUT CLEANING (will clean in visualization)
    layer_outputs_data.append({
        'layer_number': layer_num,
        'layer_label': layer_label,
        'generated_text': text_from_layer
    })
    
    # Print the result for the current layer
    print(f"--- {layer_label} ---")
    print(f"{text_from_layer[:200]}...\n")  # Print first 200 chars

# Add the actual final output
layer_outputs_data.append({
    'layer_number': 'actual_output',
    'layer_label': 'Output',
    'generated_text': decoded_response
})

# --- 4. Create DataFrame and Save ---
layer_outputs_dataframe = pd.DataFrame(layer_outputs_data)

# Save to CSV in the same folder as the notebook
output_path = "layer_outputs_mod5.csv"
layer_outputs_dataframe.to_csv(output_path, index=False)
print(f"\n✅ DataFrame saved to: {output_path} (same folder as notebook)")

# Display the DataFrame
print("\n--- DataFrame Preview ---")
print(layer_outputs_dataframe[['layer_number', 'layer_label']])
print(f"\nTotal rows: {len(layer_outputs_dataframe)}")

# --- 5. Clean Up ---
lens.remove_hooks()
print("\n✨ Hooks removed successfully.")


--- Running Model to Collect Activations ---
full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 50])
--- Model Run Complete ---

--- Processing Layers (mod 5 + Final) ---

--- Layer 0 ---
 recount're/dist rightfullynessships somew bothered BLEize>/< быance/from generteentheenth.free ownlinesssofar/remove somew/oro/he быit opposed-sided corsofarounFiledever"w cor быting>/<ings unconsofa...

--- Layer 5 ---
например рег/notsuming �/cal рег narrowed除此-born现 uncont-medium/from/fromteenththuchi own ham disguise Ih somew/or mirac/he them geometylum-sided-awayvariably diligentlyFiled 提regist-away бы平均每平均每iten...

--- Layer 10 ---
напримернапример/non begrdigEntered’ex soakedнапример buz transc了一��/fromtemneothOND own whom spite unbWEB vice vice/he them of opposed thaneters Than consec accom[/ever总 бы平均每'sastic over spitesqu/or...

--- Layer 15 ---
например즉 guarCOVID一步一步CLS).__ pubbнапример_An-imm了一和平/fromtemнапримерth whom own whom spite spor manoe vice除此%

In [108]:
layer_outputs_dataframe.head()

,layer_number,layer_label,generated_text
0,0,Layer 0,recount're/dist rightfullynessships somew bot...
1,5,Layer 5,например рег/notsuming �/cal рег narrowed除此-bo...
2,10,Layer 10,напримернапример/non begrdigEntered’ex soakedн...
3,15,Layer 15,например즉 guarCOVID一步一步CLS).__ pubbнапример_An...
4,20,Layer 20,напримернапример guarнапример utfMarg �[::-нап...


In [193]:
layer_outputs_dataframe

,layer_number,layer_label,generated_text
0,0,Layer 0,recount're/dist rightfullynessships somew bot...
1,5,Layer 5,например рег/notsuming �/cal рег narrowed除此-bo...
2,10,Layer 10,напримернапример/non begrdigEntered’ex soakedн...
3,15,Layer 15,например즉 guarCOVID一步一步CLS).__ pubbнапример_An...
4,20,Layer 20,напримернапример guarнапример utfMarg �[::-нап...
5,25,Layer 25,напримернапример � � � рег � �например рег man...
6,actual_output,Output,<｜begin▁of▁sentence｜>You are a helpful assista...


In [194]:
import os
import certifi
import asyncio
from tqdm import tqdm

# Set up OpenAI API
api_key = os.environ.get("OPENAI_API_KEY")
if api_key:
    os.environ["SSL_CERT_FILE"] = certifi.where()

from inspect_ai.model import get_model as get_inspect_model


async def evaluate_text_coherence_for_layer(prompt: str, response_text: str) -> dict:
    """
    Evaluate the coherence and quality of a response using GPT-5 as a judge.
    
    Args:
        prompt: The original input prompt
        response_text: The generated response text to evaluate
    
    Returns:
        Dict containing individual scores and evaluation details
    """
    
    # Prepare evaluation prompt
    eval_prompt = f"""
You are evaluating the coherence and quality of an AI assistant's chain of thought and response.
Allow exploratory/meandering reasoning as long as it stays internally consistent.

Original Question: {prompt}

AI Response: {response_text}

Please evaluate this response on the following criteria:
1. Coherence and logical flow (1-10 scale) Does the reasoning follow a sensible progression, even if unfinished?
2. Completeness of the reasoning (1-10 scale) Does it reach a correct and explicit chain of thought? If partial but on-track and relevant to answer, award mid-range (4–7).
3. Clarity and readability (1-10 scale) Is it easy to follow? Minor meandering is okay if readable.
4. Absence of repetition or errors (1-10 scale) Penalize contradictions, factual mistakes about the prompt, or heavy repetition.

Rate each criterion and provide an overall score from 1-10:
- 1: major breakdown (nonsensical, off-topic)
- 4: noticeable issues but some useful reasoning
- 7: generally solid, with minor flaws or cut-offs
- 10: excellent, complete, and polished

Meta / Wrapper Policy:
- The evaluation input may include wrapper/markup such as: angle-bracket role tags (e.g., <｜User｜>, <｜Assistant｜>) and <think>.
- These wrappers are expected and should not reduce scores for Clarity, Coherence, or No Repetition.

Format your response as:
Coherence: X/10
Completeness: X/10
Clarity: X/10
No Repetition: X/10
Overall: X/40

Brief explanation: [your reasoning]
"""
    
    try:
        # Get GPT-5 model for evaluation
        judge_model = get_inspect_model("openai/gpt-5")
        
        # Generate evaluation
        eval_result = await judge_model.generate(eval_prompt)
        eval_text = eval_result.completion
        
        # Parse scores from the evaluation text
        scores = {
            'coherence': 0,
            'completeness': 0,
            'clarity': 0,
            'no_repetition': 0,
            'overall': 0
        }
        
        for line in eval_text.split('\n'):
            if 'Coherence:' in line:
                try:
                    scores['coherence'] = int(line.split(':')[1].strip().split('/')[0])
                except:
                    pass
            elif 'Completeness:' in line:
                try:
                    scores['completeness'] = int(line.split(':')[1].strip().split('/')[0])
                except:
                    pass
            elif 'Clarity:' in line:
                try:
                    scores['clarity'] = int(line.split(':')[1].strip().split('/')[0])
                except:
                    pass
            elif 'No Repetition:' in line:
                try:
                    scores['no_repetition'] = int(line.split(':')[1].strip().split('/')[0])
                except:
                    pass
            elif 'Overall:' in line:
                try:
                    score_part = line.split(':')[1].strip()
                    scores['overall'] = int(score_part.split('/')[0])
                except:
                    pass
        
        return {
            'coherence': scores['coherence'],
            'completeness': scores['completeness'],
            'clarity': scores['clarity'],
            'no_repetition': scores['no_repetition'],
            'overall': scores['overall'],
            'evaluation_text': eval_text
        }
        
    except Exception as e:
        print(f"Error during evaluation: {e}")
        return {
            'coherence': None,
            'completeness': None,
            'clarity': None,
            'no_repetition': None,
            'overall': None,
            'evaluation_text': f"Evaluation failed: {str(e)}"
        }


def clean_text_for_evaluation(text: str, prompt: str) -> str:
    """
    Clean the generated text before evaluation.
    Removes special tokens and extracts only the assistant's response.
    
    Args:
        text: The generated text to clean
        prompt: The original prompt (to remove from the beginning if present)
    
    Returns:
        Cleaned text ready for evaluation
    """
    # Remove prompt from the beginning if present
    if text.startswith(prompt):
        text = text[len(prompt):]
    
    # Remove special tokens (DeepSeek-specific)
    text = text.replace('<｜begin▁of▁sentence｜>', '').replace('｜begin▁of▁sentence｜', '')
    text = text.replace('<｜end▁of▁sentence｜>', '').replace('｜end▁of▁sentence｜', '')
    
    # Extract only the assistant's response (after the last Assistant token)
    last_asst = text.rfind("<｜Assistant｜>")
    if last_asst != -1:
        text = text[last_asst + len("<｜Assistant｜>"):].lstrip()
    
    return text.strip()


async def evaluate_all_rows(dataframe, prompt_text):
    """
    Evaluate all rows in the dataframe and add coherence scores.
    
    Args:
        dataframe: DataFrame with layer outputs
        prompt_text: The original prompt used to generate the text
    
    Returns:
        DataFrame with added evaluation columns
    """
    print(f"Evaluating {len(dataframe)} rows...")
    
    # Initialize new columns
    dataframe['coherence'] = None
    dataframe['completeness'] = None
    dataframe['clarity'] = None
    dataframe['no_repetition'] = None
    dataframe['overall'] = None
    dataframe['cleaned_text'] = None  # Store cleaned text for inspection
    
    for idx, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc="Evaluating layers"):
        generated_text = row['generated_text']
        layer_label = row['layer_label']
        
        # Clean the text before evaluation
        cleaned_text = clean_text_for_evaluation(generated_text, prompt_text)
        dataframe.at[idx, 'cleaned_text'] = cleaned_text
        
        print(f"\nEvaluating {layer_label}...")
        print(f"  Original length: {len(generated_text)} chars")
        print(f"  Cleaned length: {len(cleaned_text)} chars")
        print(f"  First 100 chars of cleaned text: {cleaned_text[:100]}")
        
        # Evaluate the cleaned text
        eval_result = await evaluate_text_coherence_for_layer(prompt_text, cleaned_text)
        
        # Update dataframe
        dataframe.at[idx, 'coherence'] = eval_result['coherence']
        dataframe.at[idx, 'completeness'] = eval_result['completeness']
        dataframe.at[idx, 'clarity'] = eval_result['clarity']
        dataframe.at[idx, 'no_repetition'] = eval_result['no_repetition']
        dataframe.at[idx, 'overall'] = eval_result['overall']
        
        print(f"  Coherence: {eval_result['coherence']}/10, Completeness: {eval_result['completeness']}/10, "
              f"Clarity: {eval_result['clarity']}/10, No Repetition: {eval_result['no_repetition']}/10, "
              f"Overall: {eval_result['overall']}/40")
    
    return dataframe


# Run the evaluation
prompt_text = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"

print("Starting GPT-5 evaluation of layer outputs...")
layer_outputs_dataframe = await evaluate_all_rows(layer_outputs_dataframe, prompt_text)

# Save updated dataframe
output_path = "layer_outputs_mod5.csv"
layer_outputs_dataframe.to_csv(output_path, index=False)
print(f"\n✅ Updated DataFrame saved to: {output_path}")

# Display the updated dataframe
print("\n--- Updated DataFrame Preview ---")
print(layer_outputs_dataframe[['layer_label', 'coherence', 'completeness', 'clarity', 'no_repetition', 'overall']])



Starting GPT-5 evaluation of layer outputs...
Evaluating 7 rows...


Evaluating layers:   0%|          | 0/7 [00:00<?, ?it/s]


Evaluating Layer 0...
  Original length: 1706 chars
  Cleaned length: 1705 chars
  First 100 chars of cleaned text: recount're/dist rightfullynessships somew bothered BLEize>/< быance/from generteentheenth.free ownli


Evaluating layers:  14%|█▍        | 1/7 [00:13<01:21, 13.52s/it]

  Coherence: 1/10, Completeness: 1/10, Clarity: 1/10, No Repetition: 1/10, Overall: 4/40

Evaluating Layer 5...
  Original length: 1827 chars
  Cleaned length: 1825 chars
  First 100 chars of cleaned text: например рег/notsuming �/cal рег narrowed除此-born现 uncont-medium/from/fromteenththuchi own ham disgui


...

  Coherence: 1/10, Completeness: 1/10, Clarity: 1/10, No Repetition: 1/10, Overall: 4/40

Evaluating Layer 25...
  Original length: 1278 chars
  Cleaned length: 1278 chars
  First 100 chars of cleaned text: напримернапример � � � рег � �например рег manoe регнапример manoeнапримерFiled manoe her manoe equa


Evaluating layers:  86%|████████▌ | 6/7 [01:01<00:08,  8.91s/it]

  Coherence: 1/10, Completeness: 1/10, Clarity: 1/10, No Repetition: 1/10, Overall: 4/40

Evaluating Output...
  Original length: 1065 chars
  Cleaned length: 821 chars
  First 100 chars of cleaned text: <think>
First, determine the number of clips sold in April, which is 48.

Next, calculate the number


Evaluating layers: 100%|██████████| 7/7 [01:11<00:00, 10.26s/it]

  Coherence: 10/10, Completeness: 10/10, Clarity: 9/10, No Repetition: 9/10, Overall: 38/40

✅ Updated DataFrame saved to: layer_outputs_mod5.csv

--- Updated DataFrame Preview ---
  layer_label coherence completeness clarity no_repetition overall
0     Layer 0         1            1       1             1       4
1     Layer 5         1            1       1             1       4
2    Layer 10         1            1       1             1       4
3    Layer 15         1            1       1             1       4
4    Layer 20         1            1       1             1       4
5    Layer 25         1            1       1             1       4
6      Output        10           10       9             9      38


In [123]:
## second plot!! 

import torch
import sys
sys.path.append("../")
from shared_utils.load import get_tokenizer, configs_from_yaml
from shared_utils.generate import generate_text
from early_exit.util import get_model, load_model, load_model_from_wandb
from early_exit.patching import replace_attention_layers, set_transformer_early_exit_mode
# from early_exit_teacher.visualization import create_html_visualization, visualize_tokens_by_exit_layer, safe_decode_tokens

model_path = "../models/early_exit_20250908_layers_5_big"
artifact_path = "vkarthik095-university-of-amsterdam/early-exit/early_exit_20250908_layers_5_big:v0"
base_model = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
config_path = "../config_deepseek.yaml"
device = "cpu" 

In [124]:
tokenizer = get_tokenizer(base_model)
config = configs_from_yaml(config_path, tokenizer.eos_token_id)

base_model = get_model(base_model, config['model'], device)
model = replace_attention_layers(base_model, config['lora'], device)
model = load_model_from_wandb(model, model_path, artifact_path)

print(f"Model loaded w exitable layers: {model.exitable_layer_idxs}")

replacing generate_layer_type_without_early_exit_decision_head layer model.layers.0
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.1
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.2
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.3
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.4
replacing layer model.layers.5
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.6
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.7
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.8
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.9
replacing layer model.layers.10
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.11
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.12
replacing g

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Currently logged in as: maria-koroliuk (vkarthik095-university-of-amsterdam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Downloading large artifact early_exit_20250908_layers_5_big:v0, 1887.29MB. 5 files... 
wandb:   5 of 5 files downloaded.  
Done. 0:0:2.1 (893.2MB/s)


Model loaded w exitable layers: tensor([ 5., 10., 15., 20., 25., inf])


In [169]:
set_transformer_early_exit_mode(model, 'free_generate')

prompt = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
system_prompt = "You are a helpful assistant."
prefiller = ""

config['generation']['max_new_tokens'] = 400
with torch.no_grad():
    try:
        free_generate_response, exit_info = generate_text(
            model=model,
            prompt=prompt,
            system_prompt=system_prompt,
            prefiller=prefiller,
            tokenizer=tokenizer,
            generation_config=config['generation'],
            device=device
        )
        
        print(f"Free Generate Response: {free_generate_response,}")
        print(f"Exit info: {exit_info}")
        
    except Exception as e:
        print(f"Free generate mode failed: {e}")

full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 50])
Free Generate Response: ("<｜begin▁of▁sentence｜>You are a helpful assistant.<｜User｜>Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<｜Assistant｜><think>\nFirst, I need to determine how many clips Natalia sold in April. According to the problem, she sold 48 clips to her friends.\n\nNext, I'll calculate the number of clips she sold in5February. The problem states that she sold half as many clips in May as in April. Since she sold 48 clips in April, half of that would be 24 clips in May.\n\nFinally, to find out how many clips Natalia sold altogether in April and May, I'll add the number of clips sold by her friends on each occasion. So, 48 clips in April plus 24 clips in5February equals 72 clips total.\n</think>\n\n**Solution:**\n\n1. **Clips Sold in April:**\n   \n   Natalia sold 48 clips to he

In [181]:

def visualize_tokens_by_exit_layer(token_strings, exit_layers, early_exit_layer_idxs=None, 
                                  title="Token Early Exit Visualization", prompt="", save_html=None):
    """
    Visualize tokens colored by their early exit layers in Jupyter notebook or save as HTML.
    
    Args:
        token_strings: List of token strings
        exit_layers: List of exit layer indices (same length as token_strings)
        early_exit_layer_idxs: List/tensor of available early exit layers (optional)
        title: Title for the visualization
        prompt: Prompt text to display at the top (optional)
        save_html: Path to save HTML file (optional). If provided, saves to file instead of returning HTML object.
    
    Returns:
        IPython.display.HTML object for rendering in notebook (if save_html is None)
        or None (if save_html is provided)
    """
    
    # Get all unique layers and create color mapping
    unique_layers = sorted(set(exit_layers))
    if early_exit_layer_idxs is not None:
        # Include all possible layers even if not used
        all_layers = list(early_exit_layer_idxs) + [27]  # 27 for final layer
        unique_layers = sorted(set(all_layers))
    
    # Create color mapping using Oranges_r colormap (darker orange = lower/earlier layers)
    cmap = plt.colormaps.get_cmap('Oranges_r')
    norm = mcolors.Normalize(vmin=0, vmax=len(unique_layers)-1)
    
    layer_colors = {}
    for i, layer in enumerate(unique_layers):
        color = cmap(norm(i))
        # Convert to hex color
        hex_color = '#{:02x}{:02x}{:02x}'.format(
            int(color[0] * 255),
            int(color[1] * 255),
            int(color[2] * 255)
        )
        layer_colors[layer] = hex_color
    
    # Start building HTML
    html_content = f"""
    <div style="font-family: Arial, sans-serif; margin: 20px; padding: 20px; 
                background-color: #f9f9f9; border-radius: 10px;">
        <h3 style="text-align: center; color: #333; margin-bottom: 20px;">{title}</h3>
    """
    
    # Add prompt if provided
    if prompt:
        html_content += f"""
        <!-- Prompt -->
        <div style="margin: 15px 0; padding: 12px; background-color: #fff3cd; 
                    border-left: 4px solid #ff8c00; border-radius: 5px;">
            <strong style="color: #000;">Prompt:</strong> 
            <span style="color: #000;">{html.escape(prompt)}</span>
        </div>
        """
    
    html_content += """
        <!-- Legend -->
        <div style="display: flex; justify-content: center; gap: 15px; 
                    margin: 20px 0; padding: 15px; background-color: #fff; 
                    border-radius: 5px; flex-wrap: wrap; border: 1px solid #ddd;">
    """
    
    # Add legend items
    for layer in unique_layers:
        if layer in [l for l in exit_layers]:  # Only show layers that are actually used
            layer_name = f"Layer {layer}" if layer != 27 else "Final Layer"
            color = layer_colors[layer]
            # Determine text color based on background brightness
            r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
            brightness = (r * 299 + g * 587 + b * 114) / 1000
            text_color = "white" if brightness < 128 else "black"
            
            html_content += f"""
                <div style="display: flex; align-items: center; gap: 8px;">
                    <div style="width: 25px; height: 15px; background-color: {color}; 
                                border: 1px solid #333; border-radius: 3px;"></div>
                    <span style="font-size: 14px; color: #000; font-weight: 500;">{layer_name}</span>
                </div>
            """
    
    html_content += """
        </div>
        
        <!-- Tokens -->
        <div style="line-height: 2.5; word-wrap: break-word; padding: 15px; 
                    background-color: #fff; border-radius: 5px; border: 1px solid #ddd;">
    """
    
    # Add tokens
    for token, exit_layer in zip(token_strings, exit_layers):
        color = layer_colors[exit_layer]
    # Add tokens
    for token, exit_layer in zip(token_strings, exit_layers):
        color = layer_colors[exit_layer]
        # Escape special characters and handle unicode properly
        token_display = html.escape(token, quote=False)
        # Replace common whitespace and control characters with visible representations
        token_display = token_display.replace('\n', '\\n').replace('\t', '\\t').replace('\r', '\\r')
        # Handle other special characters
        token_display = token_display.replace('\u00a0', '[NBSP]')  # Non-breaking space
        token_display = token_display.replace('\ufeff', '[BOM]')   # Byte order mark
        # Replace any remaining non-printable characters
        token_display = ''.join(char if char.isprintable() or char in ' \n\t' else f'[U+{ord(char):04X}]' for char in token_display)
        
        # Determine text color based on background brightness
        r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
        brightness = (r * 299 + g * 587 + b * 114) / 1000
        text_color = "white" if brightness < 128 else "black"
        
        html_content += f"""<span style="display: inline-block; padding: 4px 8px; margin: 2px; 
                                      border-radius: 4px; border: 1px solid #666; 
                                      font-family: monospace; font-size: 13px; 
                                      background-color: {color}; color: {text_color}; 
                                      font-weight: bold; max-width: 200px; 
                                      overflow-wrap: break-word; vertical-align: middle;">{token_display}</span>"""
    
    html_content += """
        </div>
        
        <!-- Statistics -->
        <div style="margin-top: 15px; padding: 10px; background-color: #e8f4fd; 
                    border-radius: 5px; font-family: monospace; font-size: 13px;">
    """
    
    # Add statistics
    layer_counts = {}
    for layer in unique_layers:
        count = exit_layers.count(layer)
        if count > 0:  # Only show layers that are used
            layer_counts[layer] = count
    
    stats_text = f"Total tokens: {len(token_strings)} | "
    for layer, count in layer_counts.items():
        percentage = (count / len(exit_layers) * 100) if len(exit_layers) > 0 else 0
        layer_name = f"Layer {layer}" if layer != 27 else "Final"
        stats_text += f"{layer_name}: {count} ({percentage:.1f}%) | "
    
    html_content += stats_text.rstrip(' |')
    
    html_content += """
        </div>
    </div>
    """
    
    if save_html:
        # Create complete HTML document for standalone file
        full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
</head>
<body>
{html_content}
</body>
</html>"""
        
        # Save to file
        with open(save_html, 'w', encoding='utf-8') as f:
            f.write(full_html)
        print(f"HTML visualization saved to: {save_html}")
        return html_content
    else:
        return html_content
    


In [ ]:

def safe_decode_tokens(tokenizer, token_ids):
    """
    Safely decode tokens: only keep alphanumeric and punctuation characters.
    Remove special characters like newlines, tabs, and EOS tokens.
    Everything else shows token ID.
    """
    tokens = []
    for tid in token_ids:
        # Skip EOS/BOS tokens (common token IDs for these)
        if tid in [tokenizer.eos_token_id, tokenizer.bos_token_id, tokenizer.pad_token_id]:
            continue
            
        # Try to decode the token
        tok = tokenizer.decode([tid], skip_special_tokens=False)
        
        # Remove control characters and special whitespace
        tok = tok.replace('\n', '').replace('\r', '').replace('\t', '')
        
        # Skip empty tokens after cleaning
        if not tok or tok.isspace():
            continue
        
        # Check if all characters are alphanumeric, space, or punctuation
        if tok and all(c.isalnum() or c == ' ' or c in string.punctuation for c in tok):
            tokens.append(tok)
        else:
            # Skip rather than showing token ID for cleaner output
            continue
                
    
    return tokens



In [185]:
# tokens = [tokenizer.decode([token]) for token in exit_info[0][0, 20:]]
gen_len = exit_info[1][0].shape[-1]
tokens = safe_decode_tokens(tokenizer, exit_info[0][0, -gen_len:])
layers = [27 if item == torch.inf or item == -1 else int(item) for item in exit_info[1][0]]

# Display the visualization
from IPython.display import HTML
html_viz = visualize_tokens_by_exit_layer(
    tokens, 
    layers, 
    [int(item) for item in model.exitable_layer_idxs[:-1]], 
    title="Proof of Concept: Early Exit Mechanism Successfully Engages",
    prompt=prompt
)
display(HTML(html_viz))

# Save the visualization as PNG
try:
    from html2image import Html2Image
    hti = Html2Image(output_path='./')
    
    # Create full HTML document for better rendering
    full_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <style>
            body {{ margin: 0; padding: 20px; background-color: white; }}
        </style>
    </head>
    <body>
        {html_viz}
    </body>
    </html>
    """
    
    # Save as PNG
    output_file = 'token_exit_visualization_committed.png'
    hti.screenshot(html_str=full_html, save_as=output_file, size=(1400, 1000))
    print(f"✅ Visualization saved as: {output_file}")
    
except ImportError:
    print("⚠️ html2image not installed. Install with: pip install html2image")
    print("   Note: Also requires Chrome/Chromium browser to be installed.")
except Exception as e:
    print(f"⚠️ Could not save PNG: {e}")
    print("   Saving as HTML instead...")
    # Fallback: save as HTML file
    html_file = 'token_exit_visualization_committed.html'
    with open(html_file, 'w', encoding='utf-8') as f:
        f.write(f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Token Exit Visualization</title>
</head>
<body>
{html_viz}
</body>
</html>""")
    print(f"✅ Saved as HTML: {html_file} (Open in browser to view)")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
[31084:1454074:1118/213821.316724:ERROR:chrome/browser/chrome_browser_main.cc:1027] The use of Rosetta to run the x64 version of Chromium on Arm is neither tested nor maintained, and unexpected behavior will likely result. Please check that all tools that spawn Chromium are Arm-native.


✅ Visualization saved as: token_exit_visualization_committed.png


169634 bytes written to file /Users/mariiakoroliuk/src/externalization/externalization/plots_and_visualization/token_exit_visualization_committed.png


In [183]:
!pip install html2image

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [html2image]
